# Tratamento das Bases Gerenciais

**Inputs**\
Planilhas fornecidas pelo ELAW. Gerencial, Ativos e Encerrados.

**Outputs**\
Dados tratados de cada planilha.

In [0]:
# Configura campo para que o usuário insira parâmetros

# Parametro do formato de data do arquivo. Ex: 20240423
dbutils.widgets.text("nmtabela", "")

In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for et-xmlfile from https://files.pythonhosted.org/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/250.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 153.6/250.9 kB 4.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 5.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


  Obtaining dependency information for xlsxwriter from https://files.pythonhosted.org/packages/3a/0c/3662f4a66880196a590b202f0db82d919dd2f89e99a27fadef91c4a33d41/xlsxwriter-3.2.9-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/175.3 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 143.4/175.3 kB 4.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


  Obtaining dependency information for unidecode from https://files.pythonhosted.org/packages/8f/b7/559f59d57d18b44c6d1250d2eeaa676e028b9c527431f5d0736478a73ba1/Unidecode-1.4.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/235.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 174.1/235.8 kB 5.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Caminho das pastas e arquivos

nmtabela = dbutils.widgets.get("nmtabela")
diretorio_origem = '/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_gerenciais/external/'

arquivo_consolidado = f'TRABALHISTA_GERENCIAL_(CONSOLIDADO)-{nmtabela}.xlsx'
arquivo_ativos = 'Trabalhista_Gerencial_(Ativos)-20191220.xlsx'
arquivo_encerrados = 'Trabalhista_Gerencial_(Encerrados)-20191230.xlsx'

path_consolidado = diretorio_origem + arquivo_consolidado
path_ativos = diretorio_origem + arquivo_ativos
path_encerrados = diretorio_origem + arquivo_encerrados

path_consolidado

'/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_gerenciais/external/TRABALHISTA_GERENCIAL_(CONSOLIDADO)-20260122.xlsx'

In [0]:
# Carrega as planilhas em Spark Data Frames
df_consolidado = read_excel(path_consolidado, "'TRABALHISTA'!A6")
df_ativos = read_excel(path_ativos, "A1")
df_encerrados = read_excel(path_encerrados, "A1")

In [0]:
#df_ativos.printSchema()
#df_ativos.display()

In [0]:
# Lista as colunas com datas
consolidado_data_cols = find_columns_with_word(df_consolidado, 'DATA ')
consolidado_data_cols.append('DISTRIBUIÇÃO')

ativos_data_cols = find_columns_with_word(df_ativos, 'DATA ')
ativos_data_cols.append('DISTRIBUIÇÃO')

encerrados_data_cols = find_columns_with_word(df_encerrados, 'DATA ')
ativos_data_cols.append('DISTRIBUIÇÃO')

print("consolidado_data_cols")
print(consolidado_data_cols)
print("\n")
print("ativos_data_cols")
print(ativos_data_cols)
print("\n")
print("encerrados_data_cols")
print(encerrados_data_cols)

consolidado_data_cols
['DATA REGISTRADO', 'PARTE CONTRÁRIA DATA ADMISSÃO', 'PARTE CONTRÁRIA DATA DISPENSA ', 'DATA DE ENCERRAMENTO NO TRIBUNAL', 'DATA DE REGISTRO DO ENCERRAMENTO', 'DATA DE SOLICITAÇÃO DE ENCERRAMENTO', 'DATA DA BAIXA PROVISÓRIA', 'DATA DA REATIVAÇÃO', 'DATA AUDIENCIA INICIAL', 'DATA DE REABERTURA', 'DATA DE RECEBIMENTO', 'DATA DE DESLIGAMENTO DO FUNCIONÁRIO ATIVO', 'DATA DO TRÂNSITO EM JULGADO ESOCIAL', 'DATA DO RESULTADO DO TRÂNSITO EM JULGADO - CONHECIMENTO', 'DATA DE ADMISSÃO DO TERCEIRO', 'DATA DE DEMISSÃO DO TERCEIRO', 'DISTRIBUIÇÃO']


ativos_data_cols
['DATA REGISTRADO', 'PARTE CONTRÁRIA DATA ADMISSÃO', 'PARTE CONTRÁRIA DATA DISPENSA ', 'DATA DE ENCERRAMENTO', 'DATA DE REGISTRO DO ENCERRAMENTO', 'DATA DA BAIXA PROVISÓRIA', 'PROCESSO - DATA DA REATIVAÇÃO', 'DATA AUDIENCIA INICIAL', 'DATA DE SOLICITAÇÃO DE ENCERRAMENTO', 'DATA DE REABERTURA', 'DATA DE RECEBIMENTO', 'DISTRIBUIÇÃO', 'DISTRIBUIÇÃO']


encerrados_data_cols
['DATA REGISTRADO', 'PARTE CONTRÁRIA DATA AD

In [0]:
# Converte as datas das colunas listadas
df_consolidado = convert_to_date_format(df_consolidado, consolidado_data_cols)
df_ativos = convert_to_date_format(df_ativos, ativos_data_cols)
df_encerrados = convert_to_date_format(df_encerrados, encerrados_data_cols)

In [0]:
#df_consolidado.display()

In [0]:
# Lista colunas com percentual
consolidado_percent_cols = ["RESPONSÁBILIDADE EMPRESA PERCENTUAL", "RESPONSÁBILIDADE SÓCIO PERCENTUAL"]
ativos_percent_cols = find_columns_with_word(df_ativos, 'PERCENTUAL')
encerrados_percent_cols = find_columns_with_word(df_encerrados, 'PERCENTUAL')

print("consolidado_percent_cols")
print(consolidado_percent_cols)
print("\n")
print("ativos_percent_cols")
print(ativos_percent_cols)
print("\n")
print("encerrados_data_cols")
print(encerrados_percent_cols)

consolidado_percent_cols
['RESPONSÁBILIDADE EMPRESA PERCENTUAL', 'RESPONSÁBILIDADE SÓCIO PERCENTUAL']


ativos_percent_cols
['RESPONSÁBILIDADE COMPRADOR PERCENTUAL', 'RESPONSÁBILIDADE VENDEDOR PERCENTUAL']


encerrados_data_cols
['RESPONSÁBILIDADE COMPRADOR PERCENTUAL', 'RESPONSÁBILIDADE VENDEDOR PERCENTUAL']


In [0]:
# Ajuste dos valores percetuais
df_consolidado = convert_to_float(df_consolidado, consolidado_percent_cols)
df_ativos = convert_to_float(df_ativos, ativos_percent_cols)
df_encerrados = convert_to_float(df_encerrados, encerrados_percent_cols)

In [0]:
# Ajusta os nomes das colunas
df_consolidado = adjust_column_names(df_consolidado)
df_consolidado = remove_acentos(df_consolidado)

In [0]:
df_consolidado.createOrReplaceTempView("consolidado")

In [0]:
df_consolidado.write.format("delta").mode("overwrite").saveAsTable(f"databox.juridico_comum.trab_ger_consolida_{nmtabela}")